In [ ]:
import os
import sys
import time
import glob
import math
import random
import threading
import queue
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tiktoken", "datasets"])

import torch
import torch.nn as nn
from torch.nn import functional as F
import tiktoken
from datasets import load_dataset

# ==========================================
# 1. DUAL-T4 600M PRE-TRAINING CONFIG
# ==========================================
num_gpus = torch.cuda.device_count()
print(f"Detected {num_gpus} GPU(s)")

batch_size = 3 * max(1, num_gpus)   # 3 per T4 = 6 total (fits safely in 16GB VRAM)
grad_accum_steps = 16               # Effective batch = 96 sequences = 49,152 tokens/step
block_size = 512
max_steps = 300000    # ~14.6B tokens across 2 weeks (~24 tokens/param)
warmup_steps = 2000   # Slightly smoother warmup for the longer run                 # ~7.3B tokens total (~25-28 hrs across 4 sessions)
max_lr = 4e-4
min_lr = 4e-5
save_every = 500                    # Saves full resume state every ~30-35 mins
device = "cuda" if torch.cuda.is_available() else "cpu"

enc = tiktoken.get_encoding("gpt2")
EOT_TOKEN = enc.eot_token

# ==========================================
# 2. 600M LLAMA-3 ARCHITECTURE (GQA + WEIGHT TYING)
# ==========================================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def precompute_freqs_cis(dim, end, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()
    return torch.polar(torch.ones_like(freqs), freqs)

def apply_rotary_emb(xq, xk, freqs_cis):
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = freqs_cis.unsqueeze(0).unsqueeze(2)
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)

class GroupedQueryAttention(nn.Module):
    def __init__(self, dim, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.head_dim = dim // n_heads
        self.wq = nn.Linear(dim, n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(dim, dim, bias=False)

    def forward(self, x, freqs_cis):
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim)
        q, k = apply_rotary_emb(q, k, freqs_cis)
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=2)
            v = v.repeat_interleave(self.n_rep, dim=2)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.wo(out.transpose(1, 2).contiguous().view(B, T, C))

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim // 2, dim, bias=False)
    def forward(self, x):
        gate, val = self.w1(x).chunk(2, dim=-1)
        return self.w2(F.silu(gate) * val)

class Block(nn.Module):
    def __init__(self, dim, n_heads, n_kv_heads, ffn_hidden):
        super().__init__()
        self.ln1 = RMSNorm(dim)
        self.attention = GroupedQueryAttention(dim, n_heads, n_kv_heads)
        self.ln2 = RMSNorm(dim)
        self.ffwd = FeedForward(dim, ffn_hidden)
    def forward(self, x, freqs_cis):
        x = x + self.attention(self.ln1(x), freqs_cis)
        x = x + self.ffwd(self.ln2(x))
        return x

class Llama600M(nn.Module):
    def __init__(self, vocab_size=50304, dim=1408, n_layers=26, n_heads=22, n_kv_heads=2, ffn_hidden=7680, max_seq_len=512):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, dim)
        self.blocks = nn.ModuleList([Block(dim, n_heads, n_kv_heads, ffn_hidden) for _ in range(n_layers)])
        self.ln_f = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        # Weight tying saves ~70M redundant parameters
        self.lm_head.weight = self.token_embedding_table.weight
        self.register_buffer("freqs_cis", precompute_freqs_cis(dim // n_heads, max_seq_len))
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        B, T = idx.shape
        x = self.token_embedding_table(idx)
        freqs_cis = self.freqs_cis[:T]
        for block in self.blocks:
            x = block(x, freqs_cis)
        return self.lm_head(self.ln_f(x))

raw_model = Llama600M().to(device)
total_params = sum(p.numel() for p in raw_model.parameters()) / 1e6
print(f"Initialized Llama600M ({total_params:.1f}M unique parameters)")

optimizer = torch.optim.AdamW(raw_model.parameters(), lr=max_lr, weight_decay=0.1, betas=(0.9, 0.95))
scaler = torch.amp.GradScaler('cuda')
start_step = 0

# ==========================================
# 3. AUTO-RESUME FROM PREVIOUS SESSION IF AVAILABLE
# ==========================================
resume_paths = glob.glob("/kaggle/input/**/llama_600M_ckpt.pt", recursive=True)
if len(resume_paths) > 0:
    print(f"Found previous checkpoint: {resume_paths[0]} — Resuming training!")
    ckpt = torch.load(resume_paths[0], map_location="cpu")
    raw_model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])
    start_step = ckpt["step"]
    print(f"Resumed cleanly at Step {start_step}!")

model = nn.DataParallel(raw_model) if num_gpus > 1 else raw_model

# ==========================================
# 4. ASYNC HIGH-IQ STREAM (FineWeb-Edu + Cosmopedia + Python)
# ==========================================
print("Connecting to FineWeb-Edu, Cosmopedia, and Python streams...")
edu_stream = iter(load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True))
cosmo_stream = iter(load_dataset("HuggingFaceTB/cosmopedia", "web_samples_v2", split="train", streaming=True))
code_stream = iter(load_dataset("flytech/python-codes-25k", split="train", streaming=True))

batch_queue = queue.Queue(maxsize=48)

def data_worker():
    global edu_stream, cosmo_stream, code_stream
    token_buffer = []
    while True:
        while len(token_buffer) < batch_size * (block_size + 1):
            r = random.random()
            try:
                if r < 0.60:
                    text = next(edu_stream).get("text", "")
                elif r < 0.85:
                    text = next(cosmo_stream).get("text", "")
                else:
                    ex = next(code_stream)
                    text = f"{ex.get('instruction', '')}\n{ex.get('output', '')}"
            except StopIteration:
                edu_stream = iter(load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True))
                cosmo_stream = iter(load_dataset("HuggingFaceTB/cosmopedia", "web_samples_v2", split="train", streaming=True))
                code_stream = iter(load_dataset("flytech/python-codes-25k", split="train", streaming=True))
                continue

            if text:
                token_buffer.extend(enc.encode_ordinary(text) + [EOT_TOKEN])

        x_batch, y_batch = [], []
        for _ in range(batch_size):
            chunk = token_buffer[:block_size + 1]
            token_buffer = token_buffer[block_size + 1:]
            x_batch.append(chunk[:-1])
            y_batch.append(chunk[1:])

        batch_queue.put((
            torch.tensor(x_batch, dtype=torch.long),
            torch.tensor(y_batch, dtype=torch.long)
        ))

threading.Thread(target=data_worker, daemon=True).start()

# ==========================================
# 5. PRE-TRAINING LOOP WITH FULL CHECKPOINTING
# ==========================================
def get_lr(step):
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1.0 + math.cos(math.pi * progress))

print("Prefilling initial token batches...")
first_x, first_y = batch_queue.get()
batch_queue.put((first_x, first_y))

start_time = time.time()
running_loss = 0.0
ckpt_save_path = "/kaggle/working/llama_600M_ckpt.pt"
weights_save_path = "/kaggle/working/llama_600M_pretrained.pth"

for step in range(start_step, max_steps):
    lr = get_lr(step)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    step_loss = 0.0

    for _ in range(grad_accum_steps):
        x, y = batch_queue.get()
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
            loss = loss / grad_accum_steps
        scaler.scale(loss).backward()
        step_loss += loss.item()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    running_loss += step_loss

    if (step + 1) % 100 == 0:
        elapsed = (time.time() - start_time) / 60
        tokens_seen = (step + 1) * batch_size * grad_accum_steps * block_size / 1e9
        print(f"Step {step+1}/{max_steps} ({tokens_seen:.2f}B toks) | Loss: {running_loss/100:.4f} | LR: {lr:.2e} | Session: {elapsed:.1f}m")
        running_loss = 0.0

    if (step + 1) % save_every == 0 or (step + 1) == max_steps:
        torch.save({
            "step": step + 1,
            "model": raw_model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scaler": scaler.state_dict(),
        }, ckpt_save_path)
        torch.save(raw_model.state_dict(), weights_save_path)
        print(f"--> Saved full resume checkpoint at Step {step+1} ({ckpt_save_path})")

print("600M Pre-training Complete!")